# Task-Based fMRI Analysis: Statistical Modeling with the General Linear Model

> **Repository:** [https://github.com/ArunimGuchait/neuroimaging-intro](https://github.com/ArunimGuchait/neuroimaging-intro)  
> **Open in Colab:** [Launch this notebook](https://colab.research.google.com/github/ArunimGuchait/neuroimaging-intro/blob/main/task_based_fmri_analysis.ipynb)

This notebook is **Chapter 03** in the neuroimaging tutorial series. It teaches you how to analyze **task-based fMRI data** using statistical modeling—specifically the **General Linear Model (GLM)**—to find which brain areas activate during a task.

**Prerequisites:** This assumes you have completed [Chapter 01 (Python Basics)](https://github.com/ArunimGuchait/neuroimaging-intro/blob/main/introduction_python_for_neuroimaging.ipynb) and [Chapter 02 (Neuroimaging Basics)](https://github.com/ArunimGuchait/neuroimaging-intro/blob/main/introduction_neuroimaging_analysis.ipynb). We build on that foundation.

---

## What You Will Learn Here

1. **Task-based fMRI**: What it is and how it differs from resting-state.
2. **Experimental design**: How tasks are structured (blocks, events, conditions).
3. **The General Linear Model (GLM)**: The workhorse of fMRI statistics.
4. **Hemodynamic Response Function (HRF)**: Why brain responses are delayed and how we model that.
5. **Design matrices**: How we represent our experimental design mathematically.
6. **Contrasts**: How to compare conditions (e.g., task vs rest).
7. **Statistical maps**: Creating and interpreting z-score and t-statistic brain maps.
8. **Multiple comparisons**: Why we need corrections like FDR and what they mean.

By the end, you will run a complete **first-level analysis** (single-subject GLM) from start to finish.

---

## What is Task-Based fMRI?

In **resting-state fMRI** (Chapter 02), we record brain activity while the subject does nothing specific—just rests quietly. We analyze spontaneous fluctuations and connectivity.

In **task-based fMRI**, the subject performs specific tasks (e.g., viewing images, listening to sounds, pressing buttons) while we record brain activity. We then use statistics to find which brain regions respond to those tasks.

### Why task-based fMRI?

- **Localization**: Identify which brain areas perform specific functions (e.g., auditory cortex processes sound).
- **Cognitive neuroscience**: Test hypotheses about how the brain processes information (e.g., does the prefrontal cortex activate during decision-making?).
- **Clinical applications**: Map language or motor areas before brain surgery.

### Structure of a task-based experiment

1. **Stimuli/tasks**: The subject sees, hears, or does something (e.g., listens to words).
2. **Timing recorded**: We note when each stimulus appears (onset time) and how long it lasts (duration).
3. **fMRI acquisition**: The scanner records brain activity continuously (every ~2 seconds).
4. **Analysis**: We use statistics to find brain areas where activity correlates with the task timing.

---

## How to use this notebook

- Read the explanations carefully—task-based analysis has many new concepts.
- Run each cell in order and examine the outputs.
- Experiment: try changing parameters (e.g., different contrast, smoothing amount) to see what happens.
- Don't worry if GLM seems abstract at first—we will build it up step by step with analogies.

---
## 1. Environment setup

This notebook uses the same environment as Chapter 02. If you already set up `neuro-env`, you can reuse it.

*For a detailed explanation of virtual environments, see [Chapter 01](https://github.com/ArunimGuchait/neuroimaging-intro/blob/main/introduction_python_for_neuroimaging.ipynb).*

**Quick setup (if you haven't already):**

**Option A – Google Colab:**  
If you opened this notebook in [Google Colab](https://colab.research.google.com/github/ArunimGuchait/neuroimaging-intro/blob/main/task_based_fmri_analysis.ipynb), run the **next cell** (it will detect Colab and install packages automatically).

**Option B – Local (reuse neuro-env from chapters 1-2):**  
In a terminal, from the folder containing `requirements.txt`:
```bash
# Activate the environment you created earlier
neuro-env\Scripts\activate        # Windows
source neuro-env/bin/activate     # Linux/macOS

# If you need to install/upgrade:
pip install -r requirements.txt
```

**Option C – Install packages directly:**
```bash
pip install numpy pandas matplotlib scipy nibabel nilearn jupyter
```

**What each library does** (quick refresher):
- **nibabel**: Reads NIfTI files.
- **nilearn**: fMRI analysis, datasets, plotting, GLM modeling.
- **numpy**: Numerical arrays (brain images are NumPy arrays).
- **pandas**: Tables (event timings, confounds).
- **matplotlib & scipy**: Plotting and statistics.

In [ ]:
# Install requirements: works both locally and on Google Colab.
# Run this cell once per session.

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Google Colab: install packages
    print("Google Colab detected. Installing packages...")
    import sys
    get_ipython().system(f'{sys.executable} -m pip install --quiet numpy pandas matplotlib scipy nibabel nilearn')
    print("Done. Optional: mount Google Drive to cache data across sessions.")
else:
    # Local: packages should already be installed
    import sys
    print(f"Using local Python environment: {sys.executable}")
    print("Ensure you have activated neuro-env and installed requirements.txt")
    print("If not, run: pip install -r requirements.txt")

---
## 2. Import libraries

Now let's import the tools we need. You've seen most of these in Chapter 02, but we'll add some new ones for statistical modeling.

In [ ]:
# Import the necessary libraries
import numpy as np  # Numerical arrays (from Chapter 01)
import pandas as pd  # DataFrames for tables (from Chapter 01)
import matplotlib.pyplot as plt  # Plotting (from Chapter 01)

# Neuroimaging-specific libraries (from Chapter 02)
import nibabel as nib  # Load NIfTI files
from nilearn import datasets, plotting, image  # Nilearn core functions

# New for Chapter 03: statistical modeling
from nilearn.glm.first_level import FirstLevelModel  # GLM for task fMRI
from nilearn.glm import threshold_stats_img  # Multiple comparisons correction

# Make plots appear inline
%matplotlib inline

print("All libraries imported successfully!")

---
## 3. Download the SPM Auditory Dataset

We'll use a classic task-based fMRI dataset: the **SPM auditory experiment**. This is one of the most widely used beginner datasets in neuroimaging.

### About the experiment:

- **Task**: The subject listens to blocks of spoken words, alternating with periods of silence (rest).
- **Design**: Block design—stimuli are presented in chunks (blocks) rather than brief individual events.
- **Duration**: About 6-7 minutes of scanning.
- **Subject**: One subject (this is a **first-level** or **single-subject** analysis).

### What we'll download:

1. **Functional images**: The 4D fMRI time series (brain activity over time).
2. **Anatomical image**: A high-resolution T1-weighted structural MRI for visualization.
3. **Events file**: A table with the timing of when the auditory stimuli were presented.

Nilearn's `fetch_spm_auditory()` function downloads about 100MB of data. This only happens once—subsequent runs use the cached data.

In [ ]:
# Download the dataset
print("Downloading SPM auditory dataset...")
subject_data = datasets.fetch_spm_auditory()

# inspect what we got
print("\nDataset downloaded successfully!")
print("Dataset keys:", list(subject_data.keys()))
print("\nFunctional image (4D fMRI):", subject_data.func[0])
print("Anatomical image (structural T1):", subject_data.anat)
print("Events file (task timing):", subject_data.events)

---
## 4. Inspect the functional data

Let's load and examine the 4D fMRI data. You learned how to do this in Chapter 02—we're applying those skills here.

**Reminder** (from Chapter 02):
- **NIfTI files** store brain images as multi-dimensional arrays.
- **4D fMRI data** has shape `(x, y, z, time)` where `x`, `y`, `z` are spatial dimensions (voxels in 3D space) and `time` is the number of volumes (scans) acquired.
- The **mean functional image** (average over time) gives us a static view useful for visualization.

In [ ]:
# Load the functional image
func_img = nib.load(subject_data.func[0])
func_data = func_img.get_fdata()

print("Functional data shape:", func_data.shape)
print("  → Spatial dimensions (x, y, z):", func_data.shape[:3])
print("  → Number of time points (volumes):", func_data.shape[3])

# Compute mean over time for visualization
mean_func = image.mean_img(subject_data.func[0])

# Plot the mean functional image and anatomy side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

plotting.plot_img(mean_func, title="Mean Functional Image (EPI)", axes=axes[0], colorbar=False)
plotting.plot_anat(subject_data.anat, title="Anatomical Image (T1)", axes=axes[1])

plt.tight_layout()
plt.show()

print("\nNote: The functional image has lower resolution (larger voxels) than the anatomical image.")

---
## 5. Understanding the experimental design: the events file

In task-based fMRI, the **experimental design** is crucial. We need to know **when** stimuli were presented so we can correlate brain activity with task timing.

### What is an events file?

An **events file** is a table (typically a `.tsv` or `.csv` file) with columns describing the experimental paradigm:

- **onset**: Time (in seconds) when a stimulus/condition started.
- **duration**: How long (in seconds) the stimulus lasted.
- **trial_type**: The condition or event type (e.g., "listening", "rest", "visual", "motor").

In this auditory experiment:
- **trial_type = "active"**: Subject listens to words.
- **trial_type = "rest"**: Silence (no stimulation).

This is a **block design** because stimuli are presented in extended blocks (e.g., 42 seconds of listening, 42 seconds of rest) rather than brief events.

In [ ]:
# Load and inspect the events file
events = pd.read_table(subject_data.events)

print("Events file (first 10 rows):")
print(events.head(10))
print("\nTotal number of events:", len(events))
print("Trial types:", events['trial_type'].unique())

---
## 6. Preprocessing: Spatial smoothing

Before statistical analysis, we typically apply **spatial smoothing** to the functional data. You saw this concept in Chapter 02—here's a quick reminder of why we do it:

### Why smooth?

1. **Reduce noise**: Averaging over neighboring voxels reduces random fluctuations.
2. **Improve signal-to-noise ratio**: Makes real activation patterns more detectable.
3. **Account for anatomical variability**: When comparing across subjects, smoothing helps align functional regions that may be in slightly different anatomical locations.

### How smoothing works:

- We use a **Gaussian kernel** (a 3D bell-curve) to average each voxel with its neighbors.
- **FWHM** (Full Width at Half Maximum) controls the size of the kernel (e.g., 5mm means the kernel extends about 5mm in each direction).
- **Tradeoff**: Smoothing blurs the image—we lose spatial precision but gain statistical power.

In practice, typical smoothing is 1-2 times the voxel size (e.g., 5-8mm FWHM for 3-4mm voxels).

In [ ]:
# Smooth the functional image with a 5mm FWHM Gaussian kernel
smoothed_func = image.smooth_img(subject_data.func[0], fwhm=5)

# Compare original vs smoothed
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

plotting.plot_img(mean_func, title="Original Mean Functional", axes=axes[0], colorbar=False)
plotting.plot_img(image.mean_img(smoothed_func), title="Smoothed Mean Functional (5mm FWHM)", 
                  axes=axes[1], colorbar=False)

plt.tight_layout()
plt.show()

print("Notice the smoothed image is slightly blurrier—edges are softer.")

---
## 7. The General Linear Model (GLM): Statistical modeling of task-based fMRI

Now we reach the heart of task-based fMRI analysis: the **General Linear Model (GLM)**. This is the most important statistical tool in neuroimaging.

### What is the GLM?

The GLM is a framework for modeling data as a linear combination of predictors (regressors) plus noise. It's a generalization of regression that you might have seen in statistics class.

**Simple analogy**: Imagine you want to predict someone's height from their age and gender. You might write:

$$\text{height} = \beta_1 \times \text{age} + \beta_2 \times \text{gender} + \text{error}$$

The GLM finds the best values of $\beta_1$ and $\beta_2$ (the weights) that explain the data.

### The GLM in fMRI:

For each voxel's time series (BOLD signal over time), we model:

$$\text{BOLD signal} = \beta_1 \times \text{task regressor}_1 + \beta_2 \times \text{task regressor}_2 + \ldots + \text{confounds} + \text{error}$$

- **BOLD signal**: The measured brain activity at one voxel (a 1D array of ~100-200 time points).
- **Task regressors**: Predicted BOLD response based on when tasks occurred (derived from the events file).
- **Betas ($\beta$)**: Weights that tell us how strongly the voxel responds to each condition.
- **Confounds**: Nuisance regressors (motion, drift, etc.) that we don't care about but need to account for.
- **Error**: Random noise not explained by the model.

### The key question the GLM answers:

**"Does this voxel's activity correlate with the task timing?"**

If $\beta$ for a task is significantly different from zero, that voxel is "activated" by the task.

---

### The Hemodynamic Response Function (HRF)

There's a crucial complication: **neural activity and BOLD signal are not the same**.

- **Neural activity** happens instantly when a stimulus appears.
- **BOLD signal** (blood flow) responds slowly—it peaks about 4-6 seconds **after** the neural activity and takes 15-20 seconds to return to baseline.

This delay is called the **hemodynamic response**, and we model it with the **Hemodynamic Response Function (HRF)**.

**Analogy**: imagine ringing a doorbell (stimulus) and hearing a slow, drawn-out echo (BOLD response). The HRF is the shape of that echo.

### What does the HRF look like?

It's a specific curve (usually a gamma function), rising slowly, peaking around 5 seconds, and slowly returning to baseline. Nilearn and other tools have standard HRF shapes built in (e.g., SPM's canonical HRF).

**In practice**: We **convolve** the task timing with the HRF to create task regressors that predict the BOLD response shape.

- **Task timing**: A boxcar function (0 during rest, 1 during task).
- **Convolution with HRF**: Transforms the boxcar into a smooth, delayed curve that matches the expected BOLD signal.

Don't worry if convolution sounds abstract—the GLM software does this automatically.

In [ ]:
# Set up the GLM model
# We create a FirstLevelModel object with several parameters

fmri_glm = FirstLevelModel(
    t_r=7,  # TR (repetition time): time between scans in seconds
            # For this dataset, TR = 7 seconds (one brain volume every 7 seconds)
    
    noise_model='ar1',  # Model temporal autocorrelation (adjacent time points are correlated)
                        # 'ar1' = autoregressive model of order 1 (standard choice)
    
    hrf_model='spm',  # Use SPM's canonical HRF shape
                      # This automatically convolves our task timing with the HRF
    
    drift_model='cosine',  # Model slow signal drifts (scanner warming up, subject fatigue)
                           # 'cosine' basis removes low-frequency trends
    
    high_pass=0.01,  # High-pass filter cutoff (1/100 Hz = removes drifts slower than 100 seconds)
    
    smoothing_fwhm=5  # Apply 5mm spatial smoothing during modeling
)

print("GLM model created with parameters:")
print(f"  TR = {fmri_glm.t_r} seconds")
print(f"  HRF model = {fmri_glm.hrf_model}")
print(f"  Noise model = {fmri_glm.noise_model}")
print(f"  Drift model = {fmri_glm.drift_model}")

# Fit the model to the data
# This fits the GLM to every voxel's time series
print("\nFitting GLM to functional data (this may take a minute)...")
fmri_glm = fmri_glm.fit(subject_data.func[0], events)
print("GLM fitting complete!")

# View the design matrix
# The design matrix shows all regressors (task + confounds) over time
design_matrix = fmri_glm.design_matrices_[0]

print(f"\nDesign matrix shape: {design_matrix.shape}")
print(f"  → {design_matrix.shape[0]} time points (scans)")
print(f"  → {design_matrix.shape[1]} regressors (task conditions + confounds)")

# Plot the design matrix
plotting.plot_design_matrix(design_matrix)
plt.tight_layout()
plt.show()

print("\nDesign matrix interpretation:")
print("  - Columns: Each column is a regressor (predictor)")
print("  - 'active': Predicted BOLD for auditory stimulation (convolved with HRF)")
print("  - 'rest': Predicted BOLD for rest periods")
print("  - 'drift_*': Cosine basis functions modeling slow scanner drifts")
print("  - 'constant': Baseline (intercept)")
print("\nNotice how 'active' and 'rest' have smooth curves (not boxcars)—that's the HRF convolution!")

---
## 8. Contrasts: Comparing conditions

Now that we've fitted the GLM, we have $\beta$ weights for each regressor at every voxel. But we don't usually care about individual betas—we want to **compare conditions**.

### What is a contrast?

A **contrast** is a comparison between conditions, expressed as a weighted combination of regressors.

**Example**: To test "Is there more activation during 'active' than 'rest'?", we create a contrast:

$$\text{contrast} = 1 \times \beta_{\text{active}} + (-1) \times \beta_{\text{rest}} = \beta_{\text{active}} - \beta_{\text{rest}}$$

This asks: "At each voxel, is the response to 'active' significantly greater than 'rest'?"

### Contrast as a vector

Since the design matrix has multiple regressors (active, rest, drifts, constant), we express the contrast as a vector with one weight per regressor.

For a design matrix with columns `['active', 'rest', 'drift_1', 'drift_2', 'constant']`, the contrast "active > rest" would be:

$$[1, -1, 0, 0, 0]$$

- Weight for 'active': +1
- Weight for 'rest': -1
- Weights for confounds (drifts, constant): 0 (we ignore them)

### Statistical maps

Running a contrast produces a **statistical map**: one statistic (e.g., t-value or z-score) per voxel.

- **Positive values**: Voxels where activation is stronger for the positive weight condition ('active').
- **Negative values**: Voxels where activation is stronger for the negative weight condition ('rest').
- **Zero**: No difference.

We then threshold this map to find **significantly** activated voxels.

### Why alignment matters for overlays

Images have voxel grids and affine transforms (spatial **space**). To correctly overlay a statistical map on a background image, both must be in the same space and on the same voxel grid. In this notebook the z‑map is produced in functional space and the mean functional image (mean_func) is also in functional space — resampling the z‑map to mean_func guarantees correct alignment before plotting.

If you want to overlay the z‑map on the anatomical T1 (subject_data.anat) you must first coregister/transform images into the same space (e.g., with fMRIPrep or SPM); resampling alone without coregistration will not fix mismatched spaces.

In [ ]:
# Define the contrast: active > rest
# First, let's see the regressor names in our design matrix
print("Regressors in design matrix:")
print(list(design_matrix.columns))

# Create contrast vector
# The design matrix columns are created from the trial_type values in the events file
# Let's identify which columns correspond to our task conditions
n_regressors = len(design_matrix.columns)
contrast_vector = np.zeros(n_regressors)

# Find task-related columns (not drift or constant)
# The SPM auditory dataset typically has columns for each trial type
task_columns = [col for col in design_matrix.columns if 'drift' not in col and col != 'constant']
print(f"\nTask-related columns: {task_columns}")

# For the SPM auditory dataset, there's typically just one condition column
# representing the auditory stimulation periods
# We'll create a contrast testing if this condition is significantly different from baseline

if len(task_columns) == 1:
    # Simple case: one task condition vs implicit baseline (rest)
    task_idx = list(design_matrix.columns).index(task_columns[0])
    contrast_vector[task_idx] = 1  # Test if this condition > baseline
    
    print(f"\nContrast: {task_columns[0]} > baseline")
    print("\nContrast vector:")
    for i, col in enumerate(design_matrix.columns):
        if contrast_vector[i] != 0:
            print(f"  {col}: {contrast_vector[i]}")
    
elif len(task_columns) >= 2:
    # If there are two task conditions, compare them
    # Typically this would be something like 'active' and 'rest'
    idx1 = list(design_matrix.columns).index(task_columns[0])
    idx2 = list(design_matrix.columns).index(task_columns[1])
    contrast_vector[idx1] = 1
    contrast_vector[idx2] = -1
    
    print(f"\nContrast: {task_columns[0]} > {task_columns[1]}")
    print("\nContrast vector:")
    for i, col in enumerate(design_matrix.columns):
        if contrast_vector[i] != 0:
            print(f"  {col}: {contrast_vector[i]}")
else:
    raise ValueError("No task-related columns found in design matrix!")

# Compute the contrast
print("\nComputing contrast map...")
z_map = fmri_glm.compute_contrast(contrast_vector, output_type='z_score')

# Align z_map to functional mean (same space) for correct overlay
aligned_z_map = image.resample_to_img(z_map, mean_func) 

print("Z-map computed!")
print(f"Z-map shape: {z_map.shape}")
print("This is a 3D brain volume where each voxel has a z-score.")

# Plot the unthresholded z-map on mean functional image
print("\nPlotting unthresholded z-map:")
plotting.plot_stat_map(
    aligned_z_map,
    bg_img=mean_func,
    title='Auditory Task > Baseline (raw z-scores, no threshold)',
    cut_coords=[0, -20, 10],
    display_mode='ortho'
)
plt.show()

print("\nNote: Without thresholding, we see both significant and non-significant voxels.")
print("Many voxels have z-scores near 0 (no difference). We need to threshold for significance.")

---
## 9. Multiple comparisons correction

We have a problem: we just performed **tens of thousands of statistical tests** (one per voxel). This creates a huge risk of **false positives**.

### The multiple comparisons problem

**Analogy**: Imagine flipping a coin 100 times. Even a fair coin will sometimes give 3-4 heads in a row just by chance. If you test 100 voxels and use a p-value threshold of 0.05 (5% chance of false positive), you'd expect ~5 false positives **even if no voxels are truly activated**.

In fMRI, we test 50,000-100,000 voxels. At p < 0.05, we'd expect **thousands** of false positives!

### Solutions: correction methods

We use **multiple comparisons correction** to control false positives while maintaining sensitivity to real effects.

#### Two common methods:

1. **Family-Wise Error (FWE) correction** (Bonferroni-style):
   - **Goal**: Control the probability of **any** false positive across the whole brain.
   - **Result**: Very strict—reduces false positives but also misses some real activations.
   - **When to use**: When you need to be very confident (e.g., clinical applications).

2. **False Discovery Rate (FDR) correction**:
   - **Goal**: Control the **proportion** of false positives among all detected activations.
   - **Interpretation**: If you use FDR q < 0.05, then among all voxels you call "significant", ~5% are expected to be false positives.
   - **Result**: More lenient than FWE—detects more real activations but allows some false positives.
   - **When to use**: Exploratory research, typical fMRI studies.

We'll use **FDR** here because it's the most common choice in cognitive neuroscience.

### How it works

The `threshold_stats_img` function:
1. Takes the z-map.
2. Applies FDR correction at a chosen alpha level (e.g., 0.05).
3. Returns a thresholded map showing only significantly activated voxels.

In [ ]:
# Apply FDR correction and threshold the aligned z-map
print("Applying FDR correction (alpha = 0.05)...")
# Apply thresholding to aligned_z_map (same functional space as mean_func)
thresholded_map, threshold_value = threshold_stats_img(
    aligned_z_map,
    alpha=0.05,  # FDR threshold: expect ~5% of detected voxels to be false positives
    height_control='fdr'  # Use False Discovery Rate
)

print("FDR threshold applied.")
print(f"Threshold z-value: {threshold_value:.3f}")
print("Any voxel with |z| > {:.3f} is considered significantly activated.".format(threshold_value))

# Plot the thresholded map on mean functional image (same space)
print("\nPlotting thresholded activation map:")
plotting.plot_stat_map(
    thresholded_map,
    bg_img=mean_func,
    threshold=threshold_value,
    title='Active > Rest (FDR corrected, p < 0.05)',
    cut_coords=[-10, -20, 10],
    display_mode='ortho',
    colorbar=True,
    cmap='hot'  # 'hot' colormap: black=background, red/yellow=activation
)
plt.show()

print("\n### Interpretation:")
print("- Bright areas (red/yellow): Voxels significantly more active during 'active' than 'rest'.")
print("- These should be in auditory cortex (superior temporal gyrus) since the task was listening.")
print("- Dark areas: No significant activation.")
print("\nIf you see activation in the temporal lobes near the ears, that's the auditory cortex!")

# Optional: create an interactive plot
display = plotting.view_img_on_surf(
    thresholded_map,
    threshold=threshold_value,
    surf_mesh='fsaverage',  # Standard brain surface
    title='Interactive: Active > Rest (FDR p<0.05)',
    colorbar=True,
    cmap='hot',
)

print("\nInteractive plot (if in Jupyter, you should see a 3D brain viewer below):")
display

---
## 10. Summary and next steps

Congratulations! You've completed a full **task-based fMRI analysis** from start to finish. Let's review what you learned:

| Concept | What you now know |
|---------|-------------------|
| **Task-based fMRI** | How it differs from resting-state; stimuli timing is key |
| **Experimental design** | Events files, block vs event-related designs |
| **General Linear Model (GLM)** | Modeling BOLD signal as task regressors + confounds + noise |
| **Hemodynamic Response Function (HRF)** | Why BOLD responses are delayed; how we model that with convolution |
| **Design matrices** | Visualizing regressors (task + confounds)over time |
| **Contrasts** | Comparing conditions (e.g., active - rest) with weighted combinations |
| **Statistical maps** | Z-scores or t-statistics showing activation strength |
| **Multiple comparisons** | FDR and FWE correction to control false positives |

### What you accomplished

1. Downloaded a real task-based fMRI dataset (SPM auditory).
2. Explored the functional data and experimental design (events file).
3. Applied spatial smoothing for noise reduction.
4. Set up and fit a GLM with task regressors and confounds.
5. Created a contrast to compare 'active' vs 'rest' conditions.
6. Generated statistical maps (z-scores) showing brain activation.
7. Applied FDR correction to control for multiple comparisons.
8. Visualized significant auditory cortex activation.

### What typically comes next

This was a **first-level (single-subject) analysis**. In real research, you'd continue with:

1. **Preprocess more carefully**: Motion correction, slice-time correction, coregistration, normalization (e.g., using **fMRIPrep**).
2. **Analyze multiple subjects**: Run first-level GLM for each subject.
3. **Second-level (group) analysis**: Combine subjects' contrast maps to test population-level hypotheses (e.g., one-sample t-test).
4. **Region of Interest (ROI) analysis**: Extract parameter estimates from specific brain regions.
5. **Advanced contrasts**: Interactions, parametric modulation, etc.

### Learning resources

- **Nilearn documentation**: [GLM tutorials](https://nilearn.github.io/stable/glm/index.html) and [examples](https://nilearn.github.io/stable/auto_examples/index.html#first-level-analysis)
- **SPM manual**: Classic resource for fMRI statistics [SPM12 Manual](https://www.fil.ion.ucl.ac.uk/spm/doc/manual.pdf)
- **fMRI preprocessing**: Learn about **fMRIPrep** [fmriprep.org](https://fmriprep.org)
- **Previous chapters**:
  - [Chapter 01: Python Basics](https://github.com/ArunimGuchait/neuroimaging-intro/blob/main/introduction_python_for_neuroimaging.ipynb)
  - [Chapter 02: Neuroimaging Basics](https://github.com/ArunimGuchait/neuroimaging-intro/blob/main/introduction_neuroimaging_analysis.ipynb)

### Experiment and explore

Now that you understand the pipeline, try experimenting:

- Change the smoothing FWHM (try 0, 3mm, 8mm) and see how results differ.
- Try a different contrast (e.g., 'rest > active' by swapping weights).
- Use a stricter threshold (FWE instead of FDR, or FDR with alpha=0.01).
- Download a different dataset from Nilearn (e.g., `fetch_localizer_*` datasets).

---

**Remember:** Statistical modeling in fMRI is complex, but by breaking it into steps (design → GLM → contrasts → thresholding), it becomes manageable. Don't worry if some concepts still feel abstract—they'll solidify with practice. Keep experimenting, ask questions in [GitHub Discussions](https://github.com/ArunimGuchait/neuroimaging-intro/discussions), and explore more datasets!